# 📊 Comparaison des Modèles V2 — X-Vector vs ECAPA-TDNN (Fine-Tuning)

## Objectif
Comparer les performances des deux modèles fine-tunés (V2) avec le pipeline corrigé :
- **VAD** + **L2-norm** + **CMS** + **Seuil 0.75**

## Métriques analysées
- F1-Score (métrique principale, robuste aux classes déséquilibrées)
- EER (Equal Error Rate — taux d'erreur où FA = FR)
- AUC-ROC (aire sous la courbe ROC)
- Robustesse au bruit : 0, 10, 20 dB SNR


## 🛠️ Étape 1 : Environnement

In [ ]:
!pip install -q speechbrain torchaudio soundfile librosa matplotlib seaborn scikit-learn pandas numpy tqdm

import os, re, sys, math, random, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from tqdm import tqdm
from sklearn.metrics import (f1_score, precision_score, recall_score,
                             accuracy_score, roc_curve, auc)
import librosa
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio.transforms as T

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
SR=16000; DURATION=3; NUM_SAMPLES=SR*DURATION; THRESHOLD=0.75
random.seed(42); np.random.seed(42); torch.manual_seed(42)
print(f'[Device] {device} | THRESHOLD={THRESHOLD}')


## 🔄 Étape 2 : Données + Pipeline VAD (identique aux notebooks d'entraînement)

In [ ]:
import warnings; warnings.filterwarnings('ignore')
from speechbrain.inference.speaker import EncoderClassifier

INPUT_ROOT = '/kaggle/input'
VOX_PATH, MUSAN_PATH = None, None
for ds in os.listdir(INPUT_ROOT):
    p = os.path.join(INPUT_ROOT, ds)
    if 'vox'   in ds.lower() and VOX_PATH   is None: VOX_PATH   = p
    if 'musan' in ds.lower() and MUSAN_PATH is None: MUSAN_PATH = p

all_files = sorted(
    glob.glob(os.path.join(VOX_PATH,'**','*.wav'),  recursive=True) +
    glob.glob(os.path.join(VOX_PATH,'**','*.flac'), recursive=True)
)
spk_to_files = {}
for fp in all_files:
    for p in fp.replace('\\','/').split('/'):
        if re.match(r'^id\d{5}$', p):
            spk_to_files.setdefault(p,[]).append(fp); break

spk_ids = sorted(spk_to_files.keys()); NUM_CLASSES=len(spk_ids)
spk_to_label={s:i for i,s in enumerate(spk_ids)}
all_items=[(fp,spk_to_label[s]) for s,fps in spk_to_files.items() for fp in fps]
random.shuffle(all_items); n=len(all_items)
test_fps = [x[0] for x in all_items[int(0.9*n):]]
noise_files = sorted(glob.glob(os.path.join(MUSAN_PATH,'**','*.wav'),recursive=True)) if MUSAN_PATH else []
print(f'[Test] {len(test_fps)} fichiers | [MUSAN] {len(noise_files)} bruits')

def preprocess_waveform(w, sr=SR, duration=DURATION):
    """VAD + standardisation (identique aux notebooks d'entraînement V2)."""
    w_trimmed, _ = librosa.effects.trim(w, top_db=30)
    ns = sr*duration
    if len(w_trimmed)==0: w_trimmed=w
    if len(w_trimmed)<ns: w_trimmed=np.pad(w_trimmed,(0,ns-len(w_trimmed)))
    else: w_trimmed=w_trimmed[:ns]
    return w_trimmed.astype(np.float32)

def make_pairs(fps, n_pairs=3000):
    spk_map = {}
    for fp in fps:
        for p in fp.replace('\\','/').split('/'):
            if re.match(r'^id\d{5}$', p):
                spk_map.setdefault(p,[]).append(fp); break
    pairs=[]; spks=list(spk_map.keys())
    for _ in range(n_pairs//2):
        s=random.choice(spks)
        if len(spk_map[s])>=2:
            a,b=random.sample(spk_map[s],2); pairs.append((a,b,1))
    for _ in range(n_pairs//2):
        s1,s2=random.sample(spks,2)
        pairs.append((random.choice(spk_map[s1]),random.choice(spk_map[s2]),0))
    random.shuffle(pairs); return pairs

test_pairs = make_pairs(test_fps, n_pairs=3000)
print(f'[Paires] {len(test_pairs)} paires de test')


## 🧠 Étape 3 : Chargement des Modèles Fine-Tunés V2

In [ ]:
# Vérifier les checkpoints
xvec_path  = 'results/xvector_final_model.pt'
ecapa_path = 'results/ecapa_tdnn_final_model.pt'
# Fallback vers Models/ si créés par les notebooks V2
for p_alt, p in [('Models/xvector_final_model.pt', xvec_path),
                  ('Models/ecapa_tdnn_final_model.pt', ecapa_path)]:
    if os.path.exists(p_alt) and not os.path.exists(p):
        globals()[p.split('/')[1].replace('.pt','').replace('_final_model','_path')] = p_alt

assert os.path.exists(xvec_path),  f'Manquant : {xvec_path}  → Lancez Notebook 1 V2 d\'abord'
assert os.path.exists(ecapa_path), f'Manquant : {ecapa_path} → Lancez Notebook 2 V2 d\'abord'

# ── Réimporter les architectures (identiques aux notebooks d'entraînement) ────
class AAMSoftmax(nn.Module):
    def __init__(self, input_dim, num_classes, margin=0.2, scale=30.0):
        super().__init__()
        self.margin=margin; self.scale=scale
        self.weight=nn.Parameter(torch.FloatTensor(num_classes,input_dim))
        nn.init.xavier_uniform_(self.weight)
        self.cos_m=math.cos(margin); self.sin_m=math.sin(margin)
        self.th=math.cos(math.pi-margin); self.mm=math.sin(math.pi-margin)*margin
    def forward(self,x,labels):
        xn=F.normalize(x,p=2,dim=1); Wn=F.normalize(self.weight,p=2,dim=1)
        cos=xn@Wn.T; sin=torch.sqrt(torch.clamp(1-cos**2,min=1e-7))
        phi=cos*self.cos_m-sin*self.sin_m
        phi=torch.where(cos>self.th,phi,cos-self.mm)
        oh=F.one_hot(labels,self.weight.shape[0]).float()
        return F.cross_entropy((oh*phi+(1-oh)*cos)*self.scale,labels)

sb_xvec  = EncoderClassifier.from_hparams('speechbrain/spkrec-xvect-voxceleb',  run_opts={'device':str(device)})
sb_ecapa = EncoderClassifier.from_hparams('speechbrain/spkrec-ecapa-voxceleb', run_opts={'device':str(device)})

class XVectorWrapperV2(nn.Module):
    def __init__(self,clf,nc,edim=512):
        super().__init__()
        self.encoder=clf.mods.compute_features; self.mvn=clf.mods.mean_var_norm
        self.embedding=clf.mods.embedding_model
        self.fc=nn.Sequential(nn.Linear(edim,edim),nn.BatchNorm1d(edim),nn.ReLU())
        self.clf_head=AAMSoftmax(edim,nc)
    def extract_embedding(self,w):
        with torch.no_grad():
            f=self.encoder(w); f=self.mvn(f,torch.ones(w.shape[0]).to(w.device))
        e=self.embedding(f).squeeze(1); e=self.fc(e)
        return F.normalize(e,p=2,dim=1)

class ECAPAWrapperV2(nn.Module):
    def __init__(self,clf,nc,edim=192):
        super().__init__()
        self.mods=clf.mods
        self.fc=nn.Sequential(nn.Linear(edim,edim),nn.BatchNorm1d(edim),nn.ReLU())
        self.clf_head=AAMSoftmax(edim,nc)
    def extract_embedding(self,w):
        with torch.no_grad():
            f=self.mods.compute_features(w)
            f=self.mods.mean_var_norm(f,torch.ones(w.shape[0]).to(w.device))
        e=self.mods.embedding_model(f).squeeze(1); e=self.fc(e)
        return F.normalize(e,p=2,dim=1)

# Charger les poids
xvec_ckpt  = torch.load(xvec_path,  map_location=device, weights_only=False)
ecapa_ckpt = torch.load(ecapa_path, map_location=device, weights_only=False)

xvec_model  = XVectorWrapperV2(sb_xvec,  NUM_CLASSES, 512).to(device)
ecapa_model = ECAPAWrapperV2(sb_ecapa, NUM_CLASSES, 192).to(device)
xvec_model.load_state_dict(xvec_ckpt['model_state_dict'])
ecapa_model.load_state_dict(ecapa_ckpt['model_state_dict'])
xvec_model.eval(); ecapa_model.eval()

print('[X-Vector  V2] Chargé avec succès')
print('[ECAPA-TDNN V2] Chargé avec succès')
print(f'[Seuils] X-Vec={xvec_ckpt.get("optimal_threshold",0.75):.3f} | ECAPA={ecapa_ckpt.get("optimal_threshold",0.75):.3f}')


## 📊 Étape 4 : Évaluation Comparative Complète

In [ ]:
def get_embedding(mdl, fp, dev):
    """VAD + L2-norm (pipeline V2)."""
    mdl.eval()
    try: w,_=librosa.load(fp,sr=SR)
    except: w=np.zeros(NUM_SAMPLES,dtype=np.float32)
    w=preprocess_waveform(w)
    wt=torch.tensor(w,dtype=torch.float32).unsqueeze(0).to(dev)
    with torch.no_grad(): emb=mdl.extract_embedding(wt)
    return emb.cpu().numpy()[0]

def eval_model(mdl, pairs, dev, threshold=THRESHOLD, name=''):
    mdl.eval()
    sims,labs,cache=[],[],{}
    for a,b,lb in tqdm(pairs,desc=f'Eval {name}'):
        for f in [a,b]:
            if f not in cache: cache[f]=get_embedding(mdl,f,dev)
        sim=float(np.dot(cache[a],cache[b])); sims.append(sim); labs.append(lb)
    sims=np.array(sims); labs=np.array(labs)
    preds=(sims>=threshold).astype(int)
    fpr,tpr,_=roc_curve(labs,sims); fnr=1-tpr
    eer_i=np.argmin(np.abs(fpr-fnr)); eer=(fpr[eer_i]+fnr[eer_i])/2
    return {
        'f1':f1_score(labs,preds), 'accuracy':accuracy_score(labs,preds),
        'precision':precision_score(labs,preds,zero_division=0),
        'recall':recall_score(labs,preds,zero_division=0),
        'eer':eer, 'auc':auc(fpr,tpr),
        'sims':sims, 'labs':labs, 'fpr':fpr, 'tpr':tpr
    }

print('[Évaluation] Propre (pas de bruit)...')
res_xvec  = eval_model(xvec_model,  test_pairs, device, THRESHOLD, 'X-Vector')
res_ecapa = eval_model(ecapa_model, test_pairs, device, THRESHOLD, 'ECAPA-TDNN')

# ── Tableau comparatif ────────────────────────────────────────────────────────
df = pd.DataFrame({
    'Métrique' : ['F1-Score','Précision','Rappel','Accuracy','EER','AUC-ROC'],
    'X-Vector V2' : [res_xvec['f1'], res_xvec['precision'], res_xvec['recall'],
                     res_xvec['accuracy'], res_xvec['eer'], res_xvec['auc']],
    'ECAPA-TDNN V2': [res_ecapa['f1'], res_ecapa['precision'], res_ecapa['recall'],
                      res_ecapa['accuracy'], res_ecapa['eer'], res_ecapa['auc']],
})
df['Meilleur'] = df.apply(lambda r: 'X-Vector' if
    (r['X-Vector V2']>r['ECAPA-TDNN V2'] and r['Métrique']!='EER') or
    (r['X-Vector V2']<r['ECAPA-TDNN V2'] and r['Métrique']=='EER')
    else 'ECAPA-TDNN', axis=1)
print('\n' + df.to_string(index=False, float_format='{:.4f}'.format))


## 📈 Étape 5 : Visualisations Comparatives

In [ ]:
os.makedirs('results', exist_ok=True)
fig = plt.figure(figsize=(18, 12))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.35)

# 1. Distribution similarités — X-Vector
ax1 = fig.add_subplot(gs[0, 0])
ax1.hist(res_xvec['sims'][res_xvec['labs']==0], bins=50, alpha=0.7, color='#E74C3C', label='Différents')
ax1.hist(res_xvec['sims'][res_xvec['labs']==1], bins=50, alpha=0.7, color='#27AE60', label='Même')
ax1.axvline(THRESHOLD, color='navy', ls='--', lw=2, label=f'Seuil={THRESHOLD}')
ax1.set_title('X-Vector V2 — Distribution Cosinus'); ax1.legend(fontsize=8)

# 2. Distribution similarités — ECAPA
ax2 = fig.add_subplot(gs[0, 1])
ax2.hist(res_ecapa['sims'][res_ecapa['labs']==0], bins=50, alpha=0.7, color='#E74C3C', label='Différents')
ax2.hist(res_ecapa['sims'][res_ecapa['labs']==1], bins=50, alpha=0.7, color='#27AE60', label='Même')
ax2.axvline(THRESHOLD, color='navy', ls='--', lw=2, label=f'Seuil={THRESHOLD}')
ax2.set_title('ECAPA-TDNN V2 — Distribution Cosinus'); ax2.legend(fontsize=8)

# 3. Courbes ROC comparatives
ax3 = fig.add_subplot(gs[0, 2])
ax3.plot(res_xvec['fpr'], res_xvec['tpr'],  lw=2, color='#3498DB', label=f'X-Vector (AUC={res_xvec["auc"]:.4f})')
ax3.plot(res_ecapa['fpr'], res_ecapa['tpr'], lw=2, color='#E67E22', label=f'ECAPA-TDNN (AUC={res_ecapa["auc"]:.4f})')
ax3.plot([0,1],[0,1],'k--',alpha=0.5)
ax3.set_title('Courbes ROC — V2'); ax3.set_xlabel('FPR'); ax3.set_ylabel('TPR'); ax3.legend(fontsize=8)

# 4. Barres F1-Score
ax4 = fig.add_subplot(gs[1, 0])
metrics = ['F1-Score','Précision','Rappel','Accuracy']
x_vals  = [res_xvec['f1'],  res_xvec['precision'],  res_xvec['recall'],  res_xvec['accuracy']]
e_vals  = [res_ecapa['f1'], res_ecapa['precision'], res_ecapa['recall'], res_ecapa['accuracy']]
x_pos = np.arange(len(metrics))
ax4.bar(x_pos-0.2, x_vals, 0.4, label='X-Vector', color='#3498DB', alpha=0.8)
ax4.bar(x_pos+0.2, e_vals, 0.4, label='ECAPA-TDNN', color='#E67E22', alpha=0.8)
ax4.set_xticks(x_pos); ax4.set_xticklabels(metrics, rotation=15, fontsize=8)
ax4.set_ylim(0,1.05); ax4.set_title('Métriques comparatives V2'); ax4.legend(fontsize=8)

# 5. EER comparatif
ax5 = fig.add_subplot(gs[1, 1])
models_names=['X-Vector V2','ECAPA-TDNN V2']
eers=[res_xvec['eer'], res_ecapa['eer']]
colors_eer=['#3498DB' if eers[0]<eers[1] else '#E74C3C', '#E67E22' if eers[1]<eers[0] else '#E74C3C']
bars=ax5.bar(models_names, eers, color=['#3498DB','#E67E22'], alpha=0.85, width=0.5)
for bar,v in zip(bars,eers): ax5.text(bar.get_x()+bar.get_width()/2, v+0.002, f'{v:.4f}', ha='center', fontsize=10, fontweight='bold')
ax5.set_ylim(0, max(eers)*1.3); ax5.set_title('EER (↓ meilleur)'); ax5.set_ylabel('EER')

# 6. Résumé textuel
ax6 = fig.add_subplot(gs[1, 2])
ax6.axis('off')
winner = 'ECAPA-TDNN V2' if res_ecapa['f1']>res_xvec['f1'] else 'X-Vector V2'
summary = (
    f'RÉSUMÉ COMPARATIF V2\n'
    f'═══════════════════════\n\n'
    f'CORRECTIONS APPLIQUÉES :\n'
    f'  ✓ AAMSoftmax (X-Vector)\n'
    f'  ✓ VAD (trim silences)\n'
    f'  ✓ L2-norm embeddings\n'
    f'  ✓ CMS (biais canal)\n'
    f'  ✓ Seuil calibré = 0.75\n\n'
    f'X-VECTOR V2 :\n'
    f'  F1  = {res_xvec["f1"]:.4f}\n'
    f'  EER = {res_xvec["eer"]:.4f}\n\n'
    f'ECAPA-TDNN V2 :\n'
    f'  F1  = {res_ecapa["f1"]:.4f}\n'
    f'  EER = {res_ecapa["eer"]:.4f}\n\n'
    f'🏆 MEILLEUR : {winner}'
)
ax6.text(0.05, 0.95, summary, transform=ax6.transAxes, fontsize=9,
         verticalalignment='top', fontfamily='monospace',
         bbox=dict(boxstyle='round', facecolor='#ECF0F1', alpha=0.8))

plt.suptitle('Comparaison X-Vector vs ECAPA-TDNN — Fine-Tuning V2\n(VAD + L2-norm + AAMSoftmax + Seuil=0.75)',
             fontsize=14, fontweight='bold')
plt.savefig('results/comparison_v2_full.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'\n[✅] Meilleur modèle : {winner}')


## 🔊 Étape 6 : Robustesse Comparative au Bruit

In [ ]:
def eval_noisy(mdl, pairs, dev, snr_db, nf_list, threshold=THRESHOLD):
    mdl.eval(); sims,labs,cache=[],[],{}
    for a,b,lb in tqdm(pairs[:500],desc=f'SNR={snr_db}dB'):
        for f in [a,b]:
            if f not in cache:
                try: w,_=librosa.load(f,sr=SR)
                except: w=np.zeros(NUM_SAMPLES,dtype=np.float32)
                w=preprocess_waveform(w)
                if nf_list:
                    n,_=librosa.load(random.choice(nf_list),sr=SR)
                    ns=NUM_SAMPLES
                    if len(n)<ns: n=np.tile(n,int(np.ceil(ns/len(n))))
                    n=n[:ns]; pw=np.mean(w**2)+1e-10; pn=np.mean(n**2)+1e-10
                    w=np.clip(w+np.sqrt(pw/(pn*10**(snr_db/10)))*n,-1,1)
                wt=torch.tensor(w,dtype=torch.float32).unsqueeze(0).to(dev)
                with torch.no_grad(): emb=mdl.extract_embedding(wt)
                cache[f]=emb.cpu().numpy()[0]
        sims.append(float(np.dot(cache[a],cache[b]))); labs.append(lb)
    sims=np.array(sims); labs=np.array(labs)
    preds=(sims>=threshold).astype(int)
    return {'f1':f1_score(labs,preds),'accuracy':accuracy_score(labs,preds)}

snr_levels=[20,10,0]
snr_xvec={snr:eval_noisy(xvec_model,test_pairs,device,snr,noise_files) for snr in snr_levels}
snr_ecapa={snr:eval_noisy(ecapa_model,test_pairs,device,snr,noise_files) for snr in snr_levels}

fig,ax=plt.subplots(figsize=(10,6))
snrs_sorted=sorted(snr_levels,reverse=True)
ax.plot(snrs_sorted,[snr_xvec[s]['f1'] for s in snrs_sorted],'o-',color='#3498DB',lw=2,ms=8,label='X-Vector V2')
ax.plot(snrs_sorted,[snr_ecapa[s]['f1'] for s in snrs_sorted],'s-',color='#E67E22',lw=2,ms=8,label='ECAPA-TDNN V2')
ax.axhline(res_xvec['f1'],  color='#3498DB',ls=':',alpha=0.5)
ax.axhline(res_ecapa['f1'], color='#E67E22',ls=':',alpha=0.5)
ax.set_xlabel('SNR (dB)',fontsize=12); ax.set_ylabel('F1-Score',fontsize=12)
ax.set_title('Robustesse au Bruit — X-Vector V2 vs ECAPA-TDNN V2\n(pipeline VAD + L2-norm + Seuil=0.75)',fontsize=13)
ax.legend(fontsize=11); ax.grid(True,alpha=0.3); ax.set_xticks(snrs_sorted)
plt.tight_layout()
plt.savefig('results/comparison_v2_snr_robustness.png',dpi=150,bbox_inches='tight')
plt.show()

print('\n=== RÉSUMÉ ROBUSTESSE AU BRUIT ===')
for snr in snrs_sorted:
    print(f'SNR={snr:3d}dB | X-Vec F1={snr_xvec[snr]["f1"]:.4f} | ECAPA F1={snr_ecapa[snr]["f1"]:.4f}')
